# Model V2a — New Year Effect Only

**Extends V1** — adds global New Year dip (delta_pre, delta_mid, delta_post).
No Mid West full reset (sigma) — isolates the New Year contribution.

**Likelihood:**
$$y_{i,1} \sim \mathcal{N}(\mu_{i,1},\, \tau_i)$$
$$y_{i,t} \sim \mathcal{N}\Big(\mu_{i,t} + \phi\,(y_{i,t-1} - \mu_{i,t-1}),\, \tau_i\Big), \quad t = 2,\ldots,T$$

**Mean function:**
$$\mu_{i,t} = \alpha_i + \beta_i \cos\left(\frac{2\pi t}{52}\right) + \gamma_i \sin\left(\frac{2\pi t}{52}\right) + \delta_{\text{pre}} \cdot \mathbf{1}(t \bmod 52 = 0) + \delta_{\text{mid}} \cdot \mathbf{1}(t \bmod 52 = 1) + \delta_{\text{post}} \cdot \mathbf{1}(t \bmod 52 = 2)$$

In [ ]:
from model_helpers import *

VERSION = 'v2a'
PLOT_DIR = f'../../data/models/{VERSION}/plots/'
DATA_DIR = f'../../data/models/{VERSION}/'

df_og, raw_df, regions, n_region = load_model_data(VERSION)

## Load the model parameters

In [ ]:
n_weeks = df_og.shape[1] - 1
ev = build_event_indicators(n_weeks, regions)
df_mu, df_mu_lower, df_mu_upper, phi_mean = reconstruct_mu_ny_only(raw_df, regions, n_weeks, ev)

## Preprocess
Transpose the og df into regions (cols) x time (weeks)

In [ ]:
df_og = transpose_observed(df_og)

## Make the model estimate df
Formula
y[i,t] ~ dnorm(mu[i,t] + (phi * (y[i,t-1] - mu[i,t-1])), tau[i])

In [ ]:
df_ar1 = compute_ar1_fitted(df_mu, df_og, phi_mean)
df_ar1

## Plot MU
This should look like yearly oscillations

In [ ]:
plot_mu(df_mu, df_mu_lower, df_mu_upper, PLOT_DIR + 'mu_fit.png')

## Plot model fit

In [ ]:
plot_ar1_fit(df_ar1, PLOT_DIR + 'ar1_fit.png')

## Plot the residuals

In [ ]:
df_residuals, df_std_resid = compute_residuals(df_og, df_ar1)

In [ ]:
plot_residuals_ts(df_std_resid, PLOT_DIR + 'residuals_ts.png')

### Autocorrelation of Residuals

In [ ]:
plot_residuals_acf(df_std_resid, PLOT_DIR + 'residuals_acf.png')

### Residuals vs Fitted

In [ ]:
plot_residuals_vs_fitted(df_std_resid, df_ar1, PLOT_DIR + 'residuals_vs_fitted.png')

### QQ-Plot For Residuals

In [ ]:
plot_residuals_qq(df_std_resid, PLOT_DIR + 'residuals_qq.png')

### Residual Periodogram

In [ ]:
plot_residuals_periodogram(df_std_resid, PLOT_DIR + 'residuals_periodogram.png')

# Significance Testing

### Bonferroni correction

In [ ]:
# Available via: calc_bonf(n_comparisons, one_sided=False)

### Annual cycle amplitude
$$\hat{A} = \sqrt{\beta^2 + \gamma^2}$$

- Tests whether the seasonal cycle has a practically meaningful magnitude
- $P(A_i > \varepsilon)$ where $\varepsilon = 0.5$ trolleys per 10k
- No hypothesis test — direct posterior probability statement

In [ ]:
ampl = compute_amplitude(raw_df, regions)

In [ ]:
epsilon = 0.5

results = []
for region in regions:
    a = ampl[region]
    post_p = (a > epsilon).mean()
    results.append({
        'Region':        region,
        'Mean A':        round(a.mean(), 3),
        'Median A':      round(np.median(a), 3),
        '2.5%':          round(np.quantile(a, 0.025), 3),
        '97.5%':         round(np.quantile(a, 0.975), 3),
        f'P(A > {epsilon})': round(post_p, 4),
    })
df_ampl_ind = pd.DataFrame(results)
df_ampl_ind.to_csv(DATA_DIR + 'amplitude_individual.csv', index=False)
df_ampl_ind

### Annual cycle amplitude — pairwise

In [ ]:
df_ampl_pw = pairwise_test(ampl, regions, epsilon=0.5)
df_ampl_pw.to_csv(DATA_DIR + 'amplitude_pairwise.csv', index=False)
df_ampl_pw

In [ ]:
pairwise_heatmap(df_ampl_pw, 'P(diff > 0)',
                 'Amplitude: P(Region A > Region B)',
                 PLOT_DIR + 'amplitude_pairwise.png')

### Phase cycles — flu season test
- Based on visual inspection of a distribution from Green et al. (2015), flu season in northern ireland was round 48-14 months
- This is somewhat arbitrary and can be formally inspected

In [ ]:
flu_season_weeks    = (48, 14)
prior_winter        = (4+14) / 52
flu_key = f'{flu_season_weeks[0]} < phase < {flu_season_weeks[1]}'
ci_lower_q, ci_upper_q = 0.025, 0.975

results = []
for i, region in enumerate(regions):
    b = raw_df[f'beta[{i+1}]'].values
    g = raw_df[f'gamma[{i+1}]'].values
    rad = np.arctan2(g, b)
    peak_week = ((52 / (2 * np.pi)) * rad) % 52
    in_flu = ((peak_week >= flu_season_weeks[0]) | (peak_week <= flu_season_weeks[1])).mean()
    post_odds = in_flu / (1 - in_flu) if in_flu < 1 else float('inf')
    prior_odds = prior_winter / (1 - prior_winter)
    bf = post_odds / prior_odds
    results.append({
        'Region': region,
        'Mean peak week': round(peak_week.mean(), 1),
        '2.5%': round(np.quantile(peak_week, ci_lower_q), 1),
        '97.5%': round(np.quantile(peak_week, ci_upper_q), 1),
        flu_key: round(in_flu, 4),
        'BF': round(bf, 2),
    })
df_season_phase_test = pd.DataFrame(results)
df_season_phase_test.to_csv(DATA_DIR + 'phase_flu_test.csv', index=False)
df_season_phase_test

## Alpha baseline
Global parameter summaries and credible intervals for the region intercepts.

In [ ]:
global_params = {
    'phi':       raw_df['phi'].values,
}

rows = []
for name, vals in global_params.items():
    rows.append({
        'Parameter': name,
        'Mean':      vals.mean(),
        'SD':        vals.std(),
        '2.5%':      np.quantile(vals, 0.025),
        '97.5%':     np.quantile(vals, 0.975),
    })
pd.DataFrame(rows).round(4)

In [ ]:
results = []
for i, region in enumerate(regions):
    vals = raw_df[f'alpha[{i+1}]'].values
    ci_lower, ci_upper = np.quantile(vals, [0.025, 0.975])
    results.append({
        'Region':     region,
        'Mean':       vals.mean(),
        'Median':     np.median(vals),
        'SD':         vals.std(),
        '2.5%':       ci_lower,
        '97.5%':      ci_upper,
    })
df_alpha_ind = pd.DataFrame(results).round(4)
df_alpha_ind.to_csv(DATA_DIR + 'alpha_individual.csv', index=False)
df_alpha_ind

### Alpha baseline — pairwise test
Tests on $\alpha_i$ directly (the region intercepts).

In [ ]:
alpha_samples = {regions[i]: raw_df[f'alpha[{i+1}]'].values for i in range(n_region)}
df_alpha_pw = pairwise_test(alpha_samples, regions)
df_alpha_pw.to_csv(DATA_DIR + 'alpha_pairwise.csv', index=False)
df_alpha_pw

In [ ]:
pairwise_heatmap(df_alpha_pw, 'P(diff > 0)',
                 r'Alpha ($\alpha_i$): P(Region A > Region B)',
                 PLOT_DIR + 'alpha_pairwise.png')

### New Year — delta (global)
V2a fits global `delta_pre`, `delta_mid`, `delta_post` (shared across all regions).

In [ ]:
delta_params = ['delta_pre', 'delta_mid', 'delta_post']

results = []
for param in delta_params:
    vals = raw_df[param].values
    ci_lower, ci_upper = np.quantile(vals, [0.025, 0.975])
    results.append({
        'Parameter': param,
        'Mean': vals.mean(),
        'SD': vals.std(),
        '2.5%': ci_lower,
        '97.5%': ci_upper,
    })
df_delta = pd.DataFrame(results).round(4)
df_delta.to_csv(DATA_DIR + 'delta_global.csv', index=False)
df_delta

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4), dpi=150, layout='constrained')
ax.errorbar(['Pre', 'Mid', 'Post'], df_delta['Mean'].values,
            yerr=[df_delta['Mean'].values - df_delta['2.5%'].values,
                  df_delta['97.5%'].values - df_delta['Mean'].values],
            fmt='o', capsize=5, color='#2c7bb6')
ax.axhline(0, linestyle='--', color='grey', linewidth=0.8)
ax.set_title('Global New Year Effect (delta)')
ax.set_ylabel('Posterior estimate')
ax.set_xlabel('Phase')
sns.despine(ax=ax)
fig.savefig(PLOT_DIR + 'delta_global.png', bbox_inches='tight', dpi=150)
plt.show()

# Ranking
Ranked on $\alpha_i$ (region intercept) per MCMC draw.

Pandas ranking averages ties: 1, 2, 3, 3, 5 -> 1, 2, 2.5, 2.5, 4, 5.

In [ ]:
ranked_alpha = build_ranked_alpha(raw_df, regions)

### Tables

In [ ]:
mean_ranks = ranked_alpha.mean(axis=0)
assigned_rank = mean_ranks.rank().astype(int)

rows = []
for region in ranked_alpha.columns:
    ranks = ranked_alpha[region]
    k = assigned_rank[region]
    p_k = (ranks == k).mean()
    rows.append({
        'Region': region,
        'Mean Rank': round(ranks.mean(), 2),
        'Assigned Rank': k,
        'P(rank = k)': round(p_k, 3),
    })
df_ranks = pd.DataFrame(rows)
df_ranks.to_csv(DATA_DIR + 'ranks.csv', index=False)
df_ranks

### Distributions of ranks

In [ ]:
n_samples = len(ranked_alpha)
fig, axes = plt.subplots(2, 3, figsize=(12, 6), dpi=150, layout='constrained', sharey=True)

for ax, col in zip(axes.flatten(), ranked_alpha.columns):
    counts = ranked_alpha[col].value_counts().reindex(range(1, 7), fill_value=0)
    props = counts / n_samples
    bars = ax.bar(range(1, 7), props, color='#2c7bb6', edgecolor='white')
    ax.set_title(SHORT_NAMES.get(col, col), fontsize=10)
    ax.set_xlabel('Rank')
    ax.set_xticks(range(1, 7))
fig.suptitle(r'Posterior Distribution of $\alpha_i$ Ranks')
fig.supylabel('Proportion')
fig.savefig(PLOT_DIR + 'rank_distributions.png', bbox_inches='tight', dpi=150)
plt.show()